# 06 · Análisis geoespacial interactivo con Folium
**Proyecto final – Capstone de Ciencia de Datos Aplicada (IBM)** · Autora: Jhoselyn Vera

**Objetivo:** mapear los sitios de lanzamiento, marcar cada lanzamiento según su resultado y calcular la
proximidad a costa, ferrocarril y carretera.

> Los mapas de Folium **no se ven en la vista previa de GitHub**. El notebook guarda el mapa en
> `figures/folium_map.html` y puedes tomar una captura para la presentación.

In [ ]:
BASE = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/"
from math import atan2, cos, radians, sin, sqrt

import folium
import pandas as pd
from folium.features import DivIcon
from folium.plugins import MarkerCluster, MousePosition

spacex_df = pd.read_csv(BASE + "spacex_launch_geo.csv")[["Launch Site", "Lat", "Long", "class"]]
launch_sites_df = spacex_df.groupby("Launch Site", as_index=False).first()[["Launch Site", "Lat", "Long"]]
launch_sites_df

## Mapa base y sitios de lanzamiento

In [ ]:
site_map = folium.Map(location=[29.559684888503615, -95.0830971930759], zoom_start=4)

for _, site in launch_sites_df.iterrows():
    coord = [site["Lat"], site["Long"]]
    folium.Circle(coord, radius=1000, color="#d35400", fill=True).add_child(folium.Popup(site["Launch Site"])).add_to(site_map)
    folium.map.Marker(coord, icon=DivIcon(
        icon_size=(20, 20), icon_anchor=(0, 0),
        html=f'<div style="font-size:12; color:#d35400;"><b>{site["Launch Site"]}</b></div>')).add_to(site_map)
site_map

## Marcadores de lanzamientos por resultado (verde = éxito, rojo = fallo)

In [ ]:
spacex_df["marker_color"] = spacex_df["class"].apply(lambda c: "green" if c == 1 else "red")
marker_cluster = MarkerCluster().add_to(site_map)

for _, rec in spacex_df.iterrows():
    folium.Marker([rec["Lat"], rec["Long"]],
                  icon=folium.Icon(color="white", icon_color=rec["marker_color"]),
                  popup=f'{rec["Launch Site"]}: {"éxito" if rec["class"] == 1 else "fallo"}').add_to(marker_cluster)
site_map

## Proximidad a costa, ferrocarril y carretera

1. Ejecuta la celda siguiente y mueve el cursor sobre el mapa: el plugin `MousePosition` muestra latitud y longitud.
2. Anota las coordenadas del punto más cercano (costa, ferrocarril, carretera) para un sitio y complétalas en `PROXIMITY`.
3. Ejecuta la celda de cálculo: dibuja la línea y muestra la distancia en km.

In [ ]:
MousePosition(position="topright", separator=" | ", num_digits=5,
              prefix="Lat | Lon:").add_to(site_map)
site_map

In [ ]:
# Formato: "Sitio": {"Costa": (lat, lon), "Ferrocarril": (lat, lon), "Carretera": (lat, lon)}
# Completa con las coordenadas que leas en el mapa; si está vacío no se dibuja nada.
PROXIMITY = {}

def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0  # radio de la Tierra en km
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    a = sin((lat2 - lat1) / 2) ** 2 + cos(lat1) * cos(lat2) * sin((lon2 - lon1) / 2) ** 2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

sites = launch_sites_df.set_index("Launch Site")
for site_name, points in PROXIMITY.items():
    lat0, lon0 = sites.loc[site_name, ["Lat", "Long"]]
    for label, (lat, lon) in points.items():
        d = calculate_distance(lat0, lon0, lat, lon)
        folium.Marker([lat, lon], icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
            html=f'<div style="font-size:12; color:#1f2a44;"><b>{label}: {d:.2f} km</b></div>')).add_to(site_map)
        folium.PolyLine([[lat0, lon0], [lat, lon]], weight=2, color="#1f2a44").add_to(site_map)
        print(f"{site_name} → {label}: {d:.2f} km")
site_map

In [ ]:
import os
os.makedirs("../figures", exist_ok=True)
site_map.save("../figures/folium_map.html")
print("Mapa guardado en figures/folium_map.html")